# Fig S5G — TF motif accessibility per (state × perturbation)
The stratified analysis (08_PlotTFActivity Part B): each perturbation vs same-state NTC. Nonzero FDR-significant coefficient sub-matrix (TF × condition). Columns are **grouped by cell state** (trajectory order; perturbation within), with a colored strip and state labels above and separators between blocks; the x-axis shows only the perturbation. TF rows hierarchically clustered. Symmetric 3-stop scale.

In [ ]:
from paperfig_style import *
import numpy as np, pandas as pd, matplotlib.pyplot as plt


In [ ]:
import itertools
from scipy.cluster.hierarchy import linkage, leaves_list
from matplotlib.patches import Rectangle
cf = load_matrix('TFActivity_coefFDRsig_stateXpert.csv')
cf = cf.loc[(cf != 0).any(axis=1), (cf != 0).any(axis=0)]        # all conditions with >=1 sig TF
cs = [c.split('___')[0] for c in cf.columns]
present = [s for s in STATE_ORDER if s in set(cs)]
# columns grouped by state (trajectory order); within each state, cluster the conditions
col_ord = []
for s in present:
    idx = [j for j in range(cf.shape[1]) if cs[j] == s]
    if len(idx) > 2:
        sub = leaves_list(linkage(cf.iloc[:, idx].values.T, method='complete'))
        idx = [idx[k] for k in sub]
    col_ord.extend(idx)
cf = cf.iloc[:, col_ord]
cs   = [c.split('___')[0] for c in cf.columns]
pert = [c.split('___')[1] for c in cf.columns]
ri = leaves_list(linkage(cf.values, method='complete')) if cf.shape[0] >= 3 else np.arange(cf.shape[0])
cf = cf.iloc[ri]
n = cf.shape[1]; lim = np.abs(cf.values).max()
print('nonzero sub-matrix:', cf.shape, '| states:', present)
fig = plt.figure(figsize=(max(5.5, 0.17*n + 1.8), max(3.6, 0.14*cf.shape[0] + 1.4)))
gs = fig.add_gridspec(2, 1, height_ratios=[1, 24], hspace=0.04)
axs = fig.add_subplot(gs[0]); axh = fig.add_subplot(gs[1])
im = axh.imshow(cf.values, aspect='auto', cmap=ACTIVITY_CMAP, vmin=-lim, vmax=lim)
axh.set_xticks(range(n)); axh.set_xticklabels(pert, rotation=90, fontsize=5)
axh.set_yticks(range(cf.shape[0])); axh.set_yticklabels(tf_labels(cf.index), fontsize=5)
axh.set_xlim(-0.5, n-0.5); axh.tick_params(length=0)
[s.set_visible(False) for s in axh.spines.values()]
# ---- state annotation strip above (NEPC display names) ----
axs.set_xlim(-0.5, n-0.5); axs.set_ylim(0, 1); axs.axis('off')
start = 0
for s, grp in itertools.groupby(cs):
    cnt = sum(1 for _ in grp); center = start + cnt/2 - 0.5
    axs.add_patch(Rectangle((start-0.5, 0), cnt, 1, color=STATE_COLORS.get(s, 'grey'), ec='white', lw=0.8))
    axs.text(center, 1.7, state_label(s), ha='center', va='bottom', fontsize=5, rotation=0)
    if start > 0: axh.axvline(start-0.5, color='white', lw=1.6)
    start += cnt
# legend: color -> NEPC state name (reliable reference for narrow blocks)
from matplotlib.patches import Patch
handles = [Patch(color=STATE_COLORS[s], label=state_label(s)) for s in present]
axh.legend(handles=handles, loc='upper left', bbox_to_anchor=(1.14, 1.0), fontsize=5.5,
           handlelength=1, borderpad=0.3, title='cell state', title_fontsize=6)
cb = fig.colorbar(im, ax=[axs, axh], fraction=0.03, pad=0.02); cb.set_label('coef (FDR≤0.1)')
cb.outline.set_linewidth(0.5)
savepanel(fig, 'FigS5G_StateXPert_TFActivity')
